# Strategy development

Write normal Python strategy code here, save it into `data/strategies`, validate it, run a realistic backtest, and inspect the result in the web app.

In [ ]:
from leadlag import list_sessions, load_session, save_strategy_source, load_strategy, run_backtest, run_monte_carlo

s = load_session(list_sessions('../data')[0]['id'], data_dir='../data')
s.events.filter(signal='C').count

In [ ]:
source = r'''
from leadlag import Strategy, Order

class LighterCBboFilter(Strategy):
    name = "lighter_c_bbo_filter"
    version = "2026-04-17"
    description = "Lighter on Signal C with BBO spread filter"
    params = {
        "signal": "C",
        "follower": "Lighter Perp",
        "min_magnitude": 2.0,
        "hold_ms": 30000,
        "max_spread_bps": 2.0,
        "entry_type": "market",
        "slippage_model": "half_spread",
        "fixed_slippage_bps": 1.0,
        "position_mode": "reject",
        "stop_loss_bps": None,
        "take_profit_bps": None,
    }

    def on_event(self, event, ctx):
        p = self.params
        follower = p["follower"]
        if event.signal != p["signal"]:
            return None
        if follower not in event.lagging_followers:
            return None
        if event.magnitude_sigma < p["min_magnitude"]:
            return None
        bbo = ctx.bbo.get(follower)
        if bbo and bbo.available and bbo.spread_bps is not None and bbo.spread_bps > p["max_spread_bps"]:
            return None
        return Order(
            venue=follower,
            side="buy" if event.direction > 0 else "sell",
            qty_btc=0.001,
            entry_type=p["entry_type"],
            hold_ms=p["hold_ms"],
            stop_loss_bps=p.get("stop_loss_bps"),
            take_profit_bps=p.get("take_profit_bps"),
        )
'''

path = save_strategy_source(source, '../data/strategies/lighter_c_bbo_filter.py')
path

In [ ]:
strategy = load_strategy('../data/strategies/lighter_c_bbo_filter.py')
result = run_backtest(strategy, s)
result.stats

In [ ]:
result.plot_equity(layers=True)

In [ ]:
result.plot_spread_impact()

In [ ]:
mc = run_monte_carlo(result, n=10000)
mc.summary()

In [ ]:
result.save('../data')